In [1]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [9]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
%pip install numpy
%pip install torch # Using version 2.10.0+cu128
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Import packages
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions (NO FOLLOWING BACKWARD PASS IN VALIDATION)
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")

2.10.0+cu128


In [ ]:
##### Load and create datasets #####
# Function for obtaining windowed input-label pairs (needed as our data is highly dependent on previous data)
# E.g. [x0, x1, x2] -> y       [x1, x2, x3] -> y ...
def window_data(dataset, labels, win_size=100):
    input_seq = [dataset[i:i+win_size, :] for i in range(len(dataset) - win_size)] # Get input sequence with length = window length
    seq_label = [labels[i+win_size, 0] for i in range(len(dataset) - win_size)] # Get corresponding output for each input window sequence
    return np.array(input_seq), np.array(seq_label)


# Initialisations/definitions
num_sensor_readings = 5 # Number of sensor readings
num_classes = 3 # Number of classification classes (number of emotional states to identify)

# Import data
# data = ... # IMPORT DATA HERE
# labels = ... # IMPORT DATA LABELS HERE
#>> CREATE TRAINING, VALIDATION AND TEST DATASETS HERE
#>> Should be with type float32
# train_data =
# train_labels = 
# val_data = 
# val_labels = 
# test_data = 
# test_labels = 

# Get windowed data
window_size = 200
train_inputs, train_labels = window_data(train_data, train_labels, window_size)
val_inputs, val_labels = window_data(val_data, val_labels, window_size)
test_inputs, test_labels = window_data(test_data, test_labels, window_size)

# Convert data to tensors
train_inputs_tensor = torch.tensor(train_inputs, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)#.unsqueeze(1) # nn.CrossEntropyLoss() expects integer class labels, no floats or one-hot. Also no need for unsqueeze(1) for CrossEntropyLoss, it just take labels with dim [batch size]

val_inputs_tensor = torch.tensor(val_inputs, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)#.unsqueeze(1)

test_inputs_tensor = torch.tensor(test_inputs, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)#.unsqueeze(1)

## DEBUGGING
print(f"Shape of training inputs: {train_inputs_tensor.shape}")
print(f"Shape of training labels: {train_labels_tensor.shape}")

# Build data loaders
train_dataset = TensorDataset(train_inputs_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)#True)

val_dataset = TensorDataset(val_inputs_tensor, val_labels_tensor)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)#True)

test_dataset = TensorDataset(test_inputs_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)#True)

NameError: name 'train_data' is not defined

In [ ]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=32, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        out_logits = self.fc(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        return out_logits

In [ ]:
##### Traing model #####
model = LSTMClassifier() # Define model
print(torchinfo.summary(model, input_size=(1, window_size, num_sensor_readings))) # Input: [batch size, sequence length, input size (number of sensors)]

# Train model
train(
    model,
    train_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    optim.Adam(model.parameters(), lr=0.001),
    num_epochs=10,#500
    print_loss=True,
)

Layer (type:depth-idx)                   Output Shape              Param #
LSTMClassifier                           [32, 3]                   --
├─LSTM: 1-1                              [32, 200, 16]             1,472
├─Linear: 1-2                            [32, 3]                   51
Total params: 1,523
Trainable params: 1,523
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 9.42
Input size (MB): 0.13
Forward/backward pass size (MB): 0.82
Params size (MB): 0.01
Estimated Total Size (MB): 0.95


NameError: name 'train_loader' is not defined